In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')

# Convert timestamps to timezone-aware datetimes (UTC -> Asia/Shanghai)
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Metric summaries
metric_cmdb_ids = sorted(df_metric['cmdb_id'].dropna().unique().tolist())
metric_num_cmdb = len(metric_cmdb_ids)

metric_kpi_names = sorted(df_metric['kpi_name'].dropna().unique().tolist())
metric_num_kpi = len(metric_kpi_names)

metric_top_kpis = (
    df_metric.groupby('kpi_name')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

metric_ts_min = df_metric['ts'].min()
metric_ts_max = df_metric['ts'].max()
metric_total_rows = df_metric.shape[0]

# Trace summaries
trace_cmdb_ids = sorted(df_trace['cmdb_id'].dropna().unique().tolist())
trace_num_cmdb = len(trace_cmdb_ids)

trace_names = sorted(df_trace['trace_name'].dropna().unique().tolist())
trace_num_names = len(trace_names)

trace_top_names = (
    df_trace.groupby('trace_name')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

trace_ts_min = df_trace['ts'].min()
trace_ts_max = df_trace['ts'].max()
trace_total_rows = df_trace.shape[0]

# Assemble compact summary (dict-like). Top lists are DataFrames (max 20 rows).
summary = {
    'metric': {
        'distinct_cmdb_ids': metric_cmdb_ids,
        'num_cmdb_ids': metric_num_cmdb,
        'distinct_kpi_names_sample': metric_kpi_names[:100],  # sample (up to 100) to keep compact
        'num_kpi_names': metric_num_kpi,
        'top_kpi_names_by_count': metric_top_kpis[['kpi_name', 'count']],
        'timestamp_min': metric_ts_min.isoformat() if pd.notnull(metric_ts_min) else None,
        'timestamp_max': metric_ts_max.isoformat() if pd.notnull(metric_ts_max) else None,
        'total_rows': int(metric_total_rows)
    },
    'trace': {
        'distinct_cmdb_ids': trace_cmdb_ids,
        'num_cmdb_ids': trace_num_cmdb,
        'distinct_trace_names_sample': trace_names[:100],
        'num_trace_names': trace_num_names,
        'top_trace_names_by_count': trace_top_names[['trace_name', 'count']],
        'timestamp_min': trace_ts_min.isoformat() if pd.notnull(trace_ts_min) else None,
        'timestamp_max': trace_ts_max.isoformat() if pd.notnull(trace_ts_max) else None,
        'total_rows': int(trace_total_rows)
    }
}

summary

```
Out[1]:
```
```python
summary_text = (
    "Summary of loaded telemetry files:\n\n"
    "metric.csv:\n"
    "- Distinct cmdb_id values: 56 (examples include db_001..db_013, docker_001..docker_008, os_001..os_022, redis_001..redis_012, osb_001)\n"
    "- Distinct kpi_name values: 132\n"
    "- Top 20 kpi_name by row count (kpi_name: count): "
    "Send_total:1322, Disk_io_util:1322, Incoming_network_traffic:1321, Outgoing_network_traffic:1321, Memory_used_pct:1320, "
    "Disk_rd_ios:1320, Recv_total:1319, Agent_ping:1319, Memory_available:1319, Memory_used:1319, CPU_idle_pct:1319, Disk_svctm:1319, "
    "CPU_user_time:1319, Page_po:1319, Disk_wr_ios:1318, Page_pi:1318, Processor_load_5_min:1318, Disk_rd_kbs:1318, Disk_await:1318, CPU_iowait_time:1318\n"
    "- Timestamp range (UTC+8): 2020-05-23T01:00:00+08:00 to 2020-05-23T02:00:00+08:00\n"
    "- Total rows: 88,920\n\n"
    "trace.csv:\n"
    "- Distinct cmdb_id values: 10 (docker_001..docker_008, os_021, os_022)\n"
    "- Distinct trace_name values: 76\n"
    "- Top 20 trace_name by row count (trace_name: count): "
    "trace.self.jdbc.row_count:240, trace.self.jdbc.error_rate:240, trace.self.jdbc.duration_p95:240, trace.self.jdbc.duration_mean:240, "
    "trace.self.local.row_count:240, trace.self.local.error_rate:240, trace.self.local.duration_p95:240, trace.self.local.duration_mean:240, "
    "trace.self.csf.row_count:180, trace.self.csf.error_rate:180, trace.self.csf.duration_p95:180, trace.self.csf.duration_mean:180, "
    "trace.self.flyremote.duration_mean:120, trace.self.flyremote.duration_p95:120, trace.self.flyremote.error_rate:120, trace.self.flyremote.row_count:120, "
    "trace.from_docker_003.remoteprocess.row_count:60, trace.from_docker_003.remoteprocess.error_rate:60, trace.from_docker_003.remoteprocess.duration_p95:60, trace.from_docker_003.remoteprocess.duration_mean:60\n"
    "- Timestamp range (UTC+8): 2020-05-23T01:30:00+08:00 to 2020-05-23T01:59:00+08:00\n"
    "- Total rows: 6,240\n\n"
    "Notes: All timestamps are presented in UTC+8. The summaries above are compact aggregates (no raw rows)."
)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric': {'distinct_cmdb_ids': ['db_001', 'db_002', 'db_003', 'db_004', 'db_005', 'db_006', 'db_007', 'db_008', 'db_009', 'db_010', 'db_011', 'db_012', 'db_013', 'docker_001', 'docker_002', 'docker_003', 'docker_004', 'docker_005', 'docker_006', 'docker_007', 'docker_008', 'os_001', 'os_002', 'os_003', 'os_004', 'os_005', 'os_006', 'os_007', 'os_008', 'os_009', 'os_010', 'os_011', 'os_012', 'os_013', 'os_014', 'os_015', 'os_016', 'os_017', 'os_018', 'os_019', 'os_020', 'os_021', 'os_022', 'osb_001', 'redis_001', 'redis_002', 'redis_003', 'redis_004', 'redis_005', 'redis_006', 'redis_007', 'redis_008', 'redis_009', 'redis_010', 'redis_011', 'redis_012'], 'num_cmdb_ids': 56, 'distinct_kpi_names_sample': ['ACS', 'AIOS', 'AWS', 'Agent_ping', 'Asm_Free_Tb', 'Buffers_used', 'CPU_Used_Pct', 'CPU_free_pct', 'CPU_idle_pct', 'CPU_iowait_time', 'CPU_system_time', 'CPU_user_time', 'CPU_util_pct', 'Cache_used', 'Call_Per_Sec', 'DFParaWrite_Per_Sec', 'DbTime', 'Disk_avgqu_sz', 'Disk_await', 'Disk_io_util', 'Disk_rd_ios', 'Disk_rd_kbs', 'Disk_svctm', 'Disk_wr_ios', 'Disk_wr_kbs', 'Exec_Per_Sec', 'FS_max_avail', 'FS_max_util', 'FS_total_space', 'FS_used_pct', 'FS_used_space', 'Hang', 'ICMP_ping', 'Incoming_network_traffic', 'LFParaWrite_Per_Sec', 'LFSync_Per_Sec', 'Logic_Read_Per_Sec', 'Login_Per_Sec', 'MEM_Total', 'MEM_Used', 'MEM_Used_Pct', 'MEM_real_util', 'Memory_available', 'Memory_available_pct', 'Memory_free', 'Memory_total', 'Memory_used', 'Memory_used_pct', 'New_Tbs_Free_Gb', 'New_Tbs_Used_Pct', 'Num_of_processes', 'Num_of_running_processes', 'On_Off_State', 'Outgoing_network_traffic', 'PGA_Used_Pct', 'PGA_used_total', 'Page_pi', 'Page_po', 'Physical_Read_Per_Sec', 'Proc_Used_Pct', 'Proc_User_Used_Pct', 'Processor_load_15_min', 'Processor_load_1_min', 'Processor_load_5_min', 'Received_errors_packets', 'Received_packets', 'Received_queue', 'Recv_total', 'Redis_key_count', 'Redo_Per_Sec', 'Row_Lock', 'SEQ_Used_Pct', 'SctRead_Per_Sec', 'Send_total', 'Sent_errors_packets', 'Sent_packets', 'Sent_queue', 'SeqRead_Per_Sec', 'Sess_Active', 'Sess_Connect', 'Sess_Used_Temp', 'Sess_Used_Undo', 'Session_pct', 'Shared_memory', 'Swap_used_pct', 'System_block_queue_length', 'System_wait_queue_length', 'TPS_Per_Sec', 'Tbs_Free_Gb', 'Tbs_Used_Pct', 'TempTbs_Pct', 'Total_Tbs_Size', 'UndoTbs_Pct', 'Used_Tbs_Size', 'User_Commit', 'Zombie_Process', 'app.avg_time', 'app.num', 'app.succee_num', 'app.succee_rate'], 'num_kpi_names': 132, 'top_kpi_names_by_count':                     kpi_name  count
0                 Send_total   1322
1               Disk_io_util   1322
2   Incoming_network_traffic   1321
3   Outgoing_network_traffic   1321
4            Memory_used_pct   1320
5                Disk_rd_ios   1320
6                 Recv_total   1319
7                 Agent_ping   1319
8           Memory_available   1319
9                Memory_used   1319
10              CPU_idle_pct   1319
11                Disk_svctm   1319
12             CPU_user_time   1319
13                   Page_po   1319
14               Disk_wr_ios   1318
15                   Page_pi   1318
16      Processor_load_5_min   1318
17               Disk_rd_kbs   1318
18                Disk_await   1318
19           CPU_iowait_time   1318, 'timestamp_min': '2020-05-23T01:00:00+08:00', 'timestamp_max': '2020-05-23T02:00:00+08:00', 'total_rows': 88920}, 'trace': {'distinct_cmdb_ids': ['docker_001', 'docker_002', 'docker_003', 'docker_004', 'docker_005', 'docker_006', 'docker_007', 'docker_008', 'os_021', 'os_022'], 'num_cmdb_ids': 10, 'distinct_trace_names_sample': ['trace.from_docker_001.remoteprocess.duration_mean', 'trace.from_docker_001.remoteprocess.duration_p95', 'trace.from_docker_001.remoteprocess.error_rate', 'trace.from_docker_001.remoteprocess.row_count', 'trace.from_docker_002.remoteprocess.duration_mean', 'trace.from_docker_002.remoteprocess.duration_p95', 'trace.from_docker_002.remoteprocess.error_rate', 'trace.from_docker_002.remoteprocess.row_count', 'trace.from_docker_003.remoteprocess.duration_mean', 'trace.from_docker_003.remoteprocess.duration_p95', 'trace.from_docker_003.remoteprocess.error_rate', 'trace.from_docker_003.remoteprocess.row_count', 'trace.from_docker_004.remoteprocess.duration_mean', 'trace.from_docker_004.remoteprocess.duration_p95', 'trace.from_docker_004.remoteprocess.error_rate', 'trace.from_docker_004.remoteprocess.row_count', 'trace.from_os_021.remoteprocess.duration_mean', 'trace.from_os_021.remoteprocess.duration_p95', 'trace.from_os_021.remoteprocess.error_rate', 'trace.from_os_021.remoteprocess.row_count', 'trace.from_os_022.remoteprocess.duration_mean', 'trace.from_os_022.remoteprocess.duration_p95', 'trace.from_os_022.remoteprocess.error_rate', 'trace.from_os_022.remoteprocess.row_count', 'trace.from_root.osb.duration_mean', 'trace.from_root.osb.duration_p95', 'trace.from_root.osb.error_rate', 'trace.from_root.osb.row_count', 'trace.self.csf.duration_mean', 'trace.self.csf.duration_p95', 'trace.self.csf.error_rate', 'trace.self.csf.row_count', 'trace.self.flyremote.duration_mean', 'trace.self.flyremote.duration_p95', 'trace.self.flyremote.error_rate', 'trace.self.flyremote.row_count', 'trace.self.jdbc.duration_mean', 'trace.self.jdbc.duration_p95', 'trace.self.jdbc.error_rate', 'trace.self.jdbc.row_count', 'trace.self.local.duration_mean', 'trace.self.local.duration_p95', 'trace.self.local.error_rate', 'trace.self.local.row_count', 'trace.to_docker_001.remoteprocess.duration_mean', 'trace.to_docker_001.remoteprocess.duration_p95', 'trace.to_docker_001.remoteprocess.error_rate', 'trace.to_docker_001.remoteprocess.row_count', 'trace.to_docker_002.remoteprocess.duration_mean', 'trace.to_docker_002.remoteprocess.duration_p95', 'trace.to_docker_002.remoteprocess.error_rate', 'trace.to_docker_002.remoteprocess.row_count', 'trace.to_docker_003.remoteprocess.duration_mean', 'trace.to_docker_003.remoteprocess.duration_p95', 'trace.to_docker_003.remoteprocess.error_rate', 'trace.to_docker_003.remoteprocess.row_count', 'trace.to_docker_004.remoteprocess.duration_mean', 'trace.to_docker_004.remoteprocess.duration_p95', 'trace.to_docker_004.remoteprocess.error_rate', 'trace.to_docker_004.remoteprocess.row_count', 'trace.to_docker_005.remoteprocess.duration_mean', 'trace.to_docker_005.remoteprocess.duration_p95', 'trace.to_docker_005.remoteprocess.error_rate', 'trace.to_docker_005.remoteprocess.row_count', 'trace.to_docker_006.remoteprocess.duration_mean', 'trace.to_docker_006.remoteprocess.duration_p95', 'trace.to_docker_006.remoteprocess.error_rate', 'trace.to_docker_006.remoteprocess.row_count', 'trace.to_docker_007.remoteprocess.duration_mean', 'trace.to_docker_007.remoteprocess.duration_p95', 'trace.to_docker_007.remoteprocess.error_rate', 'trace.to_docker_007.remoteprocess.row_count', 'trace.to_docker_008.remoteprocess.duration_mean', 'trace.to_docker_008.remoteprocess.duration_p95', 'trace.to_docker_008.remoteprocess.error_rate', 'trace.to_docker_008.remoteprocess.row_count'], 'num_trace_names': 76, 'top_trace_names_by_count':                                            trace_name  count
0                           trace.self.jdbc.row_count    240
1                          trace.self.jdbc.error_rate    240
2                        trace.self.jdbc.duration_p95    240
3                       trace.self.jdbc.duration_mean    240
4                          trace.self.local.row_count    240
5                         trace.self.local.error_rate    240
6                       trace.self.local.duration_p95    240
7                      trace.self.local.duration_mean    240
8                            trace.self.csf.row_count    180
9                           trace.self.csf.error_rate    180
10                        trace.self.csf.duration_p95    180
11                       trace.self.csf.duration_mean    180
12                 trace.self.flyremote.duration_mean    120
13                  trace.self.flyremote.duration_p95    120
14                    trace.self.flyremote.error_rate    120
15                     trace.self.flyremote.row_count    120
16      trace.from_docker_003.remoteprocess.row_count     60
17     trace.from_docker_003.remoteprocess.error_rate     60
18   trace.from_docker_003.remoteprocess.duration_p95     60
19  trace.from_docker_003.remoteprocess.duration_mean     60, 'timestamp_min': '2020-05-23T01:30:00+08:00', 'timestamp_max': '2020-05-23T01:59:00+08:00', 'total_rows': 6240}}```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd
import pytz

# Reuse df_metric and tz if available in the kernel; otherwise reload minimally
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# Ensure ts column exists and is timezone-aware UTC+8
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))
else:
    # convert to tz-aware Asia/Shanghai if not already
    if df_metric['ts'].dt.tz is None:
        df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))
    else:
        df_metric['ts'] = df_metric['ts'].dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# Build candidate component list: db_001..db_013, docker_001..docker_008, os_001..os_022
db_ids = [f"db_{i:03d}" for i in range(1, 14)]
docker_ids = [f"docker_{i:03d}" for i in range(1, 9)]
os_ids = [f"os_{i:03d}" for i in range(1, 23)]
candidates = set(db_ids + docker_ids + os_ids)

# Filter metric rows to candidate components only
df_cand = df_metric[df_metric['cmdb_id'].isin(candidates)].copy()

# Group by component-KPI pair and compute required aggregates
grouped = df_cand.groupby(['cmdb_id', 'kpi_name'], observed=True).agg(
    count=('value', 'size'),
    p95_value=('value', lambda s: float(np.nanpercentile(s.dropna().values, 95)) if s.dropna().size>0 else np.nan),
    timestamp_min=('ts', 'min'),
    timestamp_max=('ts', 'max')
).reset_index()

# Format timestamp columns as ISO strings to keep output compact
grouped['timestamp_min'] = grouped['timestamp_min'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)
grouped['timestamp_max'] = grouped['timestamp_max'].apply(lambda x: x.isoformat() if pd.notnull(x) else None)

# Sort by count desc and take top 20
top20_pairs = grouped.sort_values('count', ascending=False).head(20).reset_index(drop=True)

# Total number of distinct component-KPI pairs found
total_pairs = int(grouped.shape[0])

# Keep only requested columns in the output table
top20_pairs = top20_pairs[['cmdb_id', 'kpi_name', 'count', 'p95_value', 'timestamp_min', 'timestamp_max']]

top20_pairs, total_pairs

```
Out[2]:
```
```python
summary = (
    "Summary (candidate components: db_001..db_013, docker_001..docker_008, os_001..os_022):\n\n"
    "Total distinct component-KPI pairs found: 1792\n\n"
    "Top 20 component-KPI pairs by row count (cmdb_id | kpi_name | count | p95_value | timestamp_min -> timestamp_max):\n"
    "1) os_015 | Incoming_network_traffic | 61 | 0.002360 | 2020-05-23T01:00:00+08:00 -> 2020-05-23T02:00:00+08:00\n"
    "2) os_022 | Outgoing_network_traffic | 61 | 0.084110 | 2020-05-23T01:00:00+08:00 -> 2020-05-23T02:00:00+08:00\n"
    "3) db_008 | CPU_free_pct | 61 | 97.906715 | 2020-05-23T01:00:00+08:00 -> 2020-05-23T02:00:00+08:00\n"
    "4) os_006 | Send_total | 61 | 0.000471 | 2020-05-23T01:00:00+08:00 -> 2020-05-23T02:00:00+08:00\n"
    "5) os_012 | Processor_load_5_min | 61 | 0.030000 | 2020-05-23T01:00:00+08:00 -> 2020-05-23T02:00:00+08:00\n"
    "6) os_007 | Disk_rd_ios | 61 | 0.000000 | 2020-05-23T01:00:00+08:00 -> 2020-05-23T02:00:00+08:00\n"
    "7) db_007 | MEM_real_util | 61 | 82.410000 | 2020-05-23T01:00:00+08:00 -> 2020-05-23T02:00:00+08:00\n"
    "8) os_012 | Disk_io_util | 61 | 1.000000 | 2020-05-23T01:00:00+08:00 -> 2020-05-23T02:00:00+08:00\n"
    "9) db_003 | CPU_free_pct | 61 | 99.214535 | 2020-05-23T01:00:00+08:00 -> 2020-05-23T02:00:00+08:00\n"
    "10) db_001 | CPU_free_pct | 61 | 98.970054 | 2020-05-23T01:00:00+08:00 -> 2020-05-23T02:00:00+08:00\n"
    "11) os_017 | Send_total | 61 | 0.330399 | 2020-05-23T01:00:00+08:00 -> 2020-05-23T02:00:00+08:00\n"
    "12) os_008 | Send_total | 61 | 0.000685 | 2020-05-23T01:00:00+08:00 -> 2020-05-23T02:00:00+08:00\n"
    "13) os_020 | Disk_io_util | 61 | 0.000000 | 2020-05-23T01:00:00+08:00 -> 2020-05-23T02:00:00+08:00\n"
    "14) os_013 | Disk_avgqu_sz | 60 | 0.012543 | 2020-05-23T01:00:00+08:00 -> 2020-05-23T01:59:00+08:00\n"
    "15) os_011 | Disk_wr_ios | 60 | 8.000000 | 2020-05-23T01:00:00+08:00 -> 2020-05-23T01:59:00+08:00\n"
    "16) os_011 | Disk_wr_kbs | 60 | 0.060674 | 2020-05-23T01:00:00+08:00 -> 2020-05-23T01:59:00+08:00\n"
    "17) os_013 | ICMP_ping | 60 | 1.000000 | 2020-05-23T01:00:00+08:00 -> 2020-05-23T01:59:00+08:00\n"
    "18) os_011 | Recv_total | 60 | 15.891901 | 2020-05-23T01:00:00+08:00 -> 2020-05-23T01:59:00+08:00\n"
    "19) os_011 | CPU_system_time | 60 | 0.261549 | 2020-05-23T01:00:00+08:00 -> 2020-05-23T01:59:00+08:00\n"
    "20) os_011 | Processor_load_5_min | 60 | 0.015167 | 2020-05-23T01:00:00+08:00 -> 2020-05-23T01:59:00+08:00\n\n"
    "Notes: p95 values were computed globally across each pair's entire available series (no time filtering). Timestamps are shown in UTC+8. "
    "This summary lists only the requested aggregates (top 20 pairs and the total pair count)."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                  kpi_name  count  p95_value              timestamp_min              timestamp_max
0   os_015  Incoming_network_traffic     61   0.002360  2020-05-23T01:00:00+08:00  2020-05-23T02:00:00+08:00
1   os_022  Outgoing_network_traffic     61   0.084110  2020-05-23T01:00:00+08:00  2020-05-23T02:00:00+08:00
2   db_008              CPU_free_pct     61  97.906715  2020-05-23T01:00:00+08:00  2020-05-23T02:00:00+08:00
3   os_006                Send_total     61   0.000471  2020-05-23T01:00:00+08:00  2020-05-23T02:00:00+08:00
4   os_012      Processor_load_5_min     61   0.030000  2020-05-23T01:00:00+08:00  2020-05-23T02:00:00+08:00
5   os_007               Disk_rd_ios     61   0.000000  2020-05-23T01:00:00+08:00  2020-05-23T02:00:00+08:00
6   db_007             MEM_real_util     61  82.410000  2020-05-23T01:00:00+08:00  2020-05-23T02:00:00+08:00
7   os_012              Disk_io_util     61   1.000000  2020-05-23T01:00:00+08:00  2020-05-23T02:00:00+08:00
8   db_003              CPU_free_pct     61  99.214535  2020-05-23T01:00:00+08:00  2020-05-23T02:00:00+08:00
9   db_001              CPU_free_pct     61  98.970054  2020-05-23T01:00:00+08:00  2020-05-23T02:00:00+08:00
10  os_017                Send_total     61   0.330399  2020-05-23T01:00:00+08:00  2020-05-23T02:00:00+08:00
11  os_008                Send_total     61   0.000685  2020-05-23T01:00:00+08:00  2020-05-23T02:00:00+08:00
12  os_020              Disk_io_util     61   0.000000  2020-05-23T01:00:00+08:00  2020-05-23T02:00:00+08:00
13  os_013             Disk_avgqu_sz     60   0.012543  2020-05-23T01:00:00+08:00  2020-05-23T01:59:00+08:00
14  os_011               Disk_wr_ios     60   8.000000  2020-05-23T01:00:00+08:00  2020-05-23T01:59:00+08:00
15  os_011               Disk_wr_kbs     60   0.060674  2020-05-23T01:00:00+08:00  2020-05-23T01:59:00+08:00
16  os_013                 ICMP_ping     60   1.000000  2020-05-23T01:00:00+08:00  2020-05-23T01:59:00+08:00
17  os_011                Recv_total     60  15.891901  2020-05-23T01:00:00+08:00  2020-05-23T01:59:00+08:00
18  os_011           CPU_system_time     60   0.261549  2020-05-23T01:00:00+08:00  2020-05-23T01:59:00+08:00
19  os_011      Processor_load_5_min     60   0.015167  2020-05-23T01:00:00+08:00  2020-05-23T01:59:00+08:00, 1792)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd
import pytz

# Reuse df_metric if available; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure ts column is timezone-aware Asia/Shanghai
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Candidate components
db_ids = [f"db_{i:03d}" for i in range(1, 14)]
docker_ids = [f"docker_{i:03d}" for i in range(1, 9)]
os_ids = [f"os_{i:03d}" for i in range(1, 23)]
candidates = set(db_ids + docker_ids + os_ids)

# Filter full metric to candidate components for threshold computation (global)
df_cand_full = df_metric[df_metric['cmdb_id'].isin(candidates)].copy()
df_cand_full['value'] = pd.to_numeric(df_cand_full['value'], errors='coerce')

# Compute global thresholds (p95, p5) and total count per component-KPI pair
thresholds = (
    df_cand_full
    .groupby(['cmdb_id', 'kpi_name'], observed=True)['value']
    .agg(
        count='size',
        p95=lambda s: float(np.nanpercentile(s.dropna().values, 95)) if s.dropna().size > 0 else np.nan,
        p5 = lambda s: float(np.nanpercentile(s.dropna().values, 5)) if s.dropna().size > 0 else np.nan
    )
    .reset_index()
)

# Incident window (inclusive) in UTC+8
start = pd.Timestamp('2020-05-23 01:30:00', tz=tz)
end   = pd.Timestamp('2020-05-23 02:00:00', tz=tz)

# Filter metric to candidate components and window
df_window = df_cand_full[(df_cand_full['ts'] >= start) & (df_cand_full['ts'] <= end)].copy()

# Join window rows with global thresholds
df_join = df_window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Label anomalies
df_join['high_anom'] = df_join['value'] >= df_join['p95']
df_join['low_anom']  = df_join['value'] <= df_join['p5']
df_join['is_anom']   = df_join['high_anom'] | df_join['low_anom']

# Aggregate per component-KPI within the window
agg_basic = (
    df_join
    .groupby(['cmdb_id', 'kpi_name'], observed=True)
    .agg(
        anomaly_count = ('is_anom', 'sum'),
        max_value = ('value', 'max'),
        min_value = ('value', 'min'),
        p95 = ('p95', 'first'),
        p5  = ('p5', 'first')
    )
    .reset_index()
)

# Earliest anomaly timestamp per pair
ea = (
    df_join[df_join['is_anom']]
    .groupby(['cmdb_id', 'kpi_name'], observed=True)['ts']
    .min()
    .reset_index(name='earliest_anomaly_timestamp')
)

# Merge earliest timestamp
agg = agg_basic.merge(ea, on=['cmdb_id', 'kpi_name'], how='left')

# Compute severity metric per instructions
def compute_severity(row):
    p95 = row['p95']
    p5 = row['p5']
    max_v = row['max_value']
    min_v = row['min_value']
    up = np.nan
    down = np.nan
    if pd.notnull(p95) and pd.notnull(max_v):
        up = ((max_v - p95) / p95) if p95 > 0 else (max_v - p95)
    if pd.notnull(p5) and pd.notnull(min_v):
        down = ((p5 - min_v) / p5) if p5 > 0 else (p5 - min_v)
    vals = [v for v in [up, down] if not pd.isna(v)]
    return float(max(vals)) if vals else np.nan

agg['severity'] = agg.apply(compute_severity, axis=1)

# Format earliest_anomaly_timestamp as ISO strings (UTC+8) or None
agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_timestamp'].dt.tz_convert(tz).apply(lambda x: x.isoformat() if pd.notnull(x) else None)

# Keep only pairs with anomaly_count > 0
agg_anom = agg[agg['anomaly_count'] > 0].copy()

# Total number of distinct component-KPI pairs that have anomaly_count > 0
total_anom_pairs = int(agg_anom.shape[0])

# Select and order columns, sort by anomaly_count desc then severity desc, limit to top 20
result = (
    agg_anom[['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','severity','p95','p5','max_value','min_value']]
    .sort_values(['anomaly_count','severity'], ascending=[False, False])
    .reset_index(drop=True)
)

top20_result = result.head(20)

# Display compact outputs
top20_result, total_anom_pairs

```
Out[3]:
```
Summary of results (candidate components = db_001..db_013, docker_001..docker_008, os_001..os_022; incident window = 2020-05-23 01:30:00 to 2020-05-23 02:00:00 UTC+8):

Total distinct component-KPI pairs with anomaly_count > 0: 1718

Top 20 component-KPI pairs by anomaly_count (columns: cmdb_id | kpi_name | anomaly_count | earliest_anomaly_timestamp (UTC+8) | severity | p95 | p5 | max_value | min_value):

1) os_007 | Disk_rd_ios | 31 | 2020-05-23T01:30:00+08:00 | 0.000000 | 0.000000 | 0.00 | 0.000000 | 0.00  
2) os_020 | Disk_io_util | 31 | 2020-05-23T01:30:00+08:00 | 0.000000 | 0.000000 | 0.00 | 0.000000 | 0.00  
3) db_007 | SctRead_Per_Sec | 30 | 2020-05-23T01:30:00+08:00 | 294.492958 | 0.035500 | 0.00 | 10.490000 | 0.00  
4) os_018 | Disk_svctm | 30 | 2020-05-23T01:30:00+08:00 | 36.266667 | 0.300000 | 0.00 | 11.180000 | 0.00  
5) os_009 | Disk_svctm | 30 | 2020-05-23T01:30:00+08:00 | 19.000000 | 0.100000 | 0.00 | 2.000000 | 0.00  
6) os_018 | Disk_rd_kbs | 30 | 2020-05-23T01:30:00+08:00 | 4.015385 | 0.000065 | 0.00 | 0.000326 | 0.00  
7) db_012 | tnsping_result_time | 30 | 2020-05-23T01:30:00+08:00 | 3.000000 | 20.000000 | 10.00 | 80.000000 | 10.00  
8) os_009 | Disk_await | 30 | 2020-05-23T01:30:00+08:00 | 2.660000 | 0.000000 | 0.00 | 2.660000 | 0.00  
9) db_011 | tnsping_result_time | 30 | 2020-05-23T01:30:00+08:00 | 2.500000 | 20.000000 | 10.00 | 70.000000 | 10.00  
10) db_003 | Login_Per_Sec | 30 | 2020-05-23T01:30:00+08:00 | 1.538462 | 1.300000 | 1.20 | 3.300000 | 1.20  
11) os_010 | CPU_iowait_time | 30 | 2020-05-23T01:30:00+08:00 | 1.001080 | 0.000926 | 0.00 | 0.001853 | 0.00  
12) db_001 | ACS | 30 | 2020-05-23T01:30:00+08:00 | 1.000000 | 0.020000 | 0.01 | 0.020000 | 0.00  
13) db_010 | ACS | 30 | 2020-05-23T01:30:00+08:00 | 1.000000 | 0.010000 | 0.01 | 0.020000 | 0.01  
14) db_010 | tnsping_result_time | 30 | 2020-05-23T01:30:00+08:00 | 1.000000 | 20.000000 | 10.00 | 40.000000 | 10.00  
15) db_013 | ACS | 30 | 2020-05-23T01:30:00+08:00 | 1.000000 | 0.010000 | 0.01 | 0.020000 | 0.01  
16) db_009 | Sess_Active | 30 | 2020-05-23T01:30:00+08:00 | 0.526316 | 2.000000 | 1.90 | 2.100000 | 0.90  
17) os_016 | Disk_wr_ios | 30 | 2020-05-23T01:30:00+08:00 | 0.500000 | 6.000000 | 5.00 | 9.000000 | 3.00  
18) db_006 | ACS | 30 | 2020-05-23T01:30:00+08:00 | 0.500000 | 0.020000 | 0.01 | 0.030000 | 0.01  
19) db_006 | Sess_Active | 30 | 2020-05-23T01:30:00+08:00 | 0.500000 | 0.200000 | 0.10 | 0.300000 | 0.10  
20) db_007 | Sess_Active | 30 | 2020-05-23T01:30:00+08:00 | 0.500000 | 0.200000 | 0.10 | 0.300000 | 0.10

Notes:
- p95/p5 thresholds were computed globally across each pair's entire available series (no time filtering).  
- Anomaly labeling: high_anom if value >= p95, low_anom if value <= p5; anomaly_count is number of such rows within the incident window.  
- Severity is the larger of the normalized up-side ((max_value - p95)/p95) or down-side ((p5 - min_value)/p5) per the given formula.  
- All timestamps shown are UTC+8.

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id             kpi_name  anomaly_count earliest_anomaly_timestamp    severity        p95     p5  max_value  min_value
0   os_007          Disk_rd_ios             31  2020-05-23T01:30:00+08:00    0.000000   0.000000   0.00   0.000000       0.00
1   os_020         Disk_io_util             31  2020-05-23T01:30:00+08:00    0.000000   0.000000   0.00   0.000000       0.00
2   db_007      SctRead_Per_Sec             30  2020-05-23T01:30:00+08:00  294.492958   0.035500   0.00  10.490000       0.00
3   os_018           Disk_svctm             30  2020-05-23T01:30:00+08:00   36.266667   0.300000   0.00  11.180000       0.00
4   os_009           Disk_svctm             30  2020-05-23T01:30:00+08:00   19.000000   0.100000   0.00   2.000000       0.00
5   os_018          Disk_rd_kbs             30  2020-05-23T01:30:00+08:00    4.015385   0.000065   0.00   0.000326       0.00
6   db_012  tnsping_result_time             30  2020-05-23T01:30:00+08:00    3.000000  20.000000  10.00  80.000000      10.00
7   os_009           Disk_await             30  2020-05-23T01:30:00+08:00    2.660000   0.000000   0.00   2.660000       0.00
8   db_011  tnsping_result_time             30  2020-05-23T01:30:00+08:00    2.500000  20.000000  10.00  70.000000      10.00
9   db_003        Login_Per_Sec             30  2020-05-23T01:30:00+08:00    1.538462   1.300000   1.20   3.300000       1.20
10  os_010      CPU_iowait_time             30  2020-05-23T01:30:00+08:00    1.001080   0.000926   0.00   0.001853       0.00
11  db_001                  ACS             30  2020-05-23T01:30:00+08:00    1.000000   0.020000   0.01   0.020000       0.00
12  db_010                  ACS             30  2020-05-23T01:30:00+08:00    1.000000   0.010000   0.01   0.020000       0.01
13  db_010  tnsping_result_time             30  2020-05-23T01:30:00+08:00    1.000000  20.000000  10.00  40.000000      10.00
14  db_013                  ACS             30  2020-05-23T01:30:00+08:00    1.000000   0.010000   0.01   0.020000       0.01
15  db_009          Sess_Active             30  2020-05-23T01:30:00+08:00    0.526316   2.000000   1.90   2.100000       0.90
16  os_016          Disk_wr_ios             30  2020-05-23T01:30:00+08:00    0.500000   6.000000   5.00   9.000000       3.00
17  db_006                  ACS             30  2020-05-23T01:30:00+08:00    0.500000   0.020000   0.01   0.030000       0.01
18  db_006          Sess_Active             30  2020-05-23T01:30:00+08:00    0.500000   0.200000   0.10   0.300000       0.10
19  db_007          Sess_Active             30  2020-05-23T01:30:00+08:00    0.500000   0.200000   0.10   0.300000       0.10, 1718)```
```

In [ ]:
# In[4]:

import numpy as np
import pandas as pd
import pytz

# Reuse df_trace if available; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure ts column is timezone-aware Asia/Shanghai
if 'ts' not in df_trace.columns:
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # convert to tz-aware Asia/Shanghai if necessary
    if df_trace['ts'].dt.tz is None:
        df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df_trace['ts'] = df_trace['ts'].dt.tz_convert(tz)

# Ensure value numeric
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Compute global thresholds (p95, p5) and count per (cmdb_id, trace_name) using entire series
thresholds = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], observed=True)['value']
    .agg(
        count='size',
        p95=lambda s: float(np.nanpercentile(s.dropna().values, 95)) if s.dropna().size > 0 else np.nan,
        p5 = lambda s: float(np.nanpercentile(s.dropna().values, 5)) if s.dropna().size > 0 else np.nan
    )
    .reset_index()
)

# Incident window (inclusive) in UTC+8
start = pd.Timestamp('2020-05-23 01:30:00', tz=tz)
end   = pd.Timestamp('2020-05-23 02:00:00', tz=tz)

# Filter trace to window
df_window = df_trace[(df_trace['ts'] >= start) & (df_trace['ts'] <= end)].copy()

# Join with global thresholds
df_join = df_window.merge(thresholds, on=['cmdb_id', 'trace_name'], how='left')

# Label anomalies
df_join['high_anom'] = df_join['value'] >= df_join['p95']
df_join['low_anom']  = df_join['value'] <= df_join['p5']
df_join['is_anom']   = df_join['high_anom'] | df_join['low_anom']

# Aggregate per (cmdb_id, trace_name) within the window
agg_basic = (
    df_join
    .groupby(['cmdb_id', 'trace_name'], observed=True)
    .agg(
        anomaly_count = ('is_anom', 'sum'),
        max_value = ('value', 'max'),
        min_value = ('value', 'min'),
        p95 = ('p95', 'first'),
        p5  = ('p5', 'first')
    )
    .reset_index()
)

# Earliest anomaly timestamp per pair
ea = (
    df_join[df_join['is_anom']]
    .groupby(['cmdb_id', 'trace_name'], observed=True)['ts']
    .min()
    .reset_index(name='earliest_anomaly_timestamp')
)

# Merge earliest timestamp
agg = agg_basic.merge(ea, on=['cmdb_id', 'trace_name'], how='left')

# Compute severity per instructions
def compute_severity(row):
    p95 = row['p95']
    p5 = row['p5']
    max_v = row['max_value']
    min_v = row['min_value']
    up = np.nan
    down = np.nan
    if pd.notnull(p95) and pd.notnull(max_v):
        up = ((max_v - p95) / p95) if p95 > 0 else (max_v - p95)
    if pd.notnull(p5) and pd.notnull(min_v):
        down = ((p5 - min_v) / p5) if p5 > 0 else (p5 - min_v)
    vals = [v for v in [up, down] if not pd.isna(v)]
    return float(max(vals)) if vals else np.nan

agg['severity'] = agg.apply(compute_severity, axis=1)

# Format earliest_anomaly_timestamp as ISO strings (UTC+8) or None
agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_timestamp'].dt.tz_convert(tz).apply(lambda x: x.isoformat() if pd.notnull(x) else None)

# Keep only pairs with anomaly_count > 0
agg_anom = agg[agg['anomaly_count'] > 0].copy()

# Total number of distinct trace-series with anomaly_count > 0
total_anom_pairs = int(agg_anom.shape[0])

# Select and order columns, sort by anomaly_count desc then severity desc, limit to top 20
result = (
    agg_anom[['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','severity','p95','p5','max_value','min_value']]
    .sort_values(['anomaly_count','severity'], ascending=[False, False])
    .reset_index(drop=True)
)

top20_result = result.head(20)

# Display compact outputs
top20_result, total_anom_pairs

```
Out[4]:
```
```python
summary = (
    "Trace-series anomaly summary (window 2020-05-23 01:30:00 to 2020-05-23 02:00:00 UTC+8):\n\n"
    "Total distinct trace-series with anomaly_count > 0: 208\n\n"
    "Top 20 (cmdb_id | trace_name | anomaly_count | earliest_anomaly_timestamp | severity | p95 | p5 | max | min):\n"
    "1) docker_006 | trace.self.jdbc.duration_p95 | 30 | 2020-05-23T01:30:00+08:00 | 1.682927 | 0.004100 | 0.003 | 0.011000 | 0.003\n"
    "2) docker_002 | trace.self.jdbc.duration_p95 | 30 | 2020-05-23T01:30:00+08:00 | 1.333333 | 0.003000 | 0.003 | 0.007000 | 0.003\n"
    "3) docker_004 | trace.self.jdbc.duration_p95 | 30 | 2020-05-23T01:30:00+08:00 | 1.084885 | 0.003357 | 0.003 | 0.007000 | 0.003\n"
    "4) docker_005 | trace.self.jdbc.duration_p95 | 30 | 2020-05-23T01:30:00+08:00 | 0.746032 | 0.006300 | 0.003 | 0.011000 | 0.003\n"
    "5) docker_007 | trace.self.jdbc.duration_p95 | 30 | 2020-05-23T01:30:00+08:00 | 0.746032 | 0.006300 | 0.003 | 0.011000 | 0.003\n"
    "6) docker_008 | trace.self.jdbc.duration_p95 | 30 | 2020-05-23T01:30:00+08:00 | 0.739130 | 0.005750 | 0.003 | 0.010000 | 0.003\n"
    "7) docker_001 | trace.self.jdbc.duration_p95 | 30 | 2020-05-23T01:30:00+08:00 | 0.690141 | 0.003550 | 0.003 | 0.006000 | 0.003\n"
    "8) docker_003 | trace.self.flyremote.error_rate   | 30 | 2020-05-23T01:30:00+08:00 | 0.007246 | 0.000000 | 0.000 | 0.007246 | 0.000\n"
    "9) docker_001 | trace.from_os_022.remoteprocess.error_rate | 30 | 2020-05-23T01:30:00+08:00 | 0.000000 | 0.000000 | 0.000 | 0.000000 | 0.000\n"
    "10) docker_001 | trace.self.csf.error_rate        | 30 | 2020-05-23T01:30:00+08:00 | 0.000000 | 0.000000 | 0.000 | 0.000000 | 0.000\n"
    "11) docker_001 | trace.self.flyremote.error_rate  | 30 | 2020-05-23T01:30:00+08:00 | 0.000000 | 0.000000 | 0.000 | 0.000000 | 0.000\n"
    "12) docker_001 | trace.self.jdbc.error_rate       | 30 | 2020-05-23T01:30:00+08:00 | 0.000000 | 0.000000 | 0.000 | 0.000000 | 0.000\n"
    "13) docker_001 | trace.self.local.error_rate      | 30 | 2020-05-23T01:30:00+08:00 | 0.000000 | 0.000000 | 0.000 | 0.000000 | 0.000\n"
    "14) docker_001 | trace.to_docker_007.remoteprocess.error_rate | 30 | 2020-05-23T01:30:00+08:00 | 0.000000 | 0.000000 | 0.000 | 0.000000 | 0.000\n"
    "15) docker_001 | trace.to_docker_008.remoteprocess.error_rate | 30 | 2020-05-23T01:30:00+08:00 | 0.000000 | 0.000000 | 0.000 | 0.000000 | 0.000\n"
    "16) docker_002 | trace.from_os_022.remoteprocess.error_rate | 30 | 2020-05-23T01:30:00+08:00 | 0.000000 | 0.000000 | 0.000 | 0.000000 | 0.000\n"
    "17) docker_002 | trace.self.csf.error_rate        | 30 | 2020-05-23T01:30:00+08:00 | 0.000000 | 0.000000 | 0.000 | 0.000000 | 0.000\n"
    "18) docker_002 | trace.self.flyremote.error_rate  | 30 | 2020-05-23T01:30:00+08:00 | 0.000000 | 0.000000 | 0.000 | 0.000000 | 0.000\n"
    "19) docker_002 | trace.self.jdbc.error_rate       | 30 | 2020-05-23T01:30:00+08:00 | 0.000000 | 0.000000 | 0.000 | 0.000000 | 0.000\n"
    "20) docker_002 | trace.self.local.error_rate      | 30 | 2020-05-23T01:30:00+08:00 | 0.000000 | 0.000000 | 0.000 | 0.000000 | 0.000\n\n"
    "Notes:\n"
    "- p95/p5 computed globally (entire trace.csv) per (cmdb_id, trace_name).\n"
    "- Anomalies labeled where value >= p95 or value <= p5 within the incident window.\n"
    "- Severity is the larger normalized deviation above p95 or below p5 as requested.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                    trace_name  anomaly_count earliest_anomaly_timestamp  severity       p95     p5  max_value  min_value
0   docker_006                  trace.self.jdbc.duration_p95             30  2020-05-23T01:30:00+08:00  1.682927  0.004100  0.003   0.011000      0.003
1   docker_002                  trace.self.jdbc.duration_p95             30  2020-05-23T01:30:00+08:00  1.333333  0.003000  0.003   0.007000      0.003
2   docker_004                  trace.self.jdbc.duration_p95             30  2020-05-23T01:30:00+08:00  1.084885  0.003357  0.003   0.007000      0.003
3   docker_005                  trace.self.jdbc.duration_p95             30  2020-05-23T01:30:00+08:00  0.746032  0.006300  0.003   0.011000      0.003
4   docker_007                  trace.self.jdbc.duration_p95             30  2020-05-23T01:30:00+08:00  0.746032  0.006300  0.003   0.011000      0.003
5   docker_008                  trace.self.jdbc.duration_p95             30  2020-05-23T01:30:00+08:00  0.739130  0.005750  0.003   0.010000      0.003
6   docker_001                  trace.self.jdbc.duration_p95             30  2020-05-23T01:30:00+08:00  0.690141  0.003550  0.003   0.006000      0.003
7   docker_003               trace.self.flyremote.error_rate             30  2020-05-23T01:30:00+08:00  0.007246  0.000000  0.000   0.007246      0.000
8   docker_001    trace.from_os_022.remoteprocess.error_rate             30  2020-05-23T01:30:00+08:00  0.000000  0.000000  0.000   0.000000      0.000
9   docker_001                     trace.self.csf.error_rate             30  2020-05-23T01:30:00+08:00  0.000000  0.000000  0.000   0.000000      0.000
10  docker_001               trace.self.flyremote.error_rate             30  2020-05-23T01:30:00+08:00  0.000000  0.000000  0.000   0.000000      0.000
11  docker_001                    trace.self.jdbc.error_rate             30  2020-05-23T01:30:00+08:00  0.000000  0.000000  0.000   0.000000      0.000
12  docker_001                   trace.self.local.error_rate             30  2020-05-23T01:30:00+08:00  0.000000  0.000000  0.000   0.000000      0.000
13  docker_001  trace.to_docker_007.remoteprocess.error_rate             30  2020-05-23T01:30:00+08:00  0.000000  0.000000  0.000   0.000000      0.000
14  docker_001  trace.to_docker_008.remoteprocess.error_rate             30  2020-05-23T01:30:00+08:00  0.000000  0.000000  0.000   0.000000      0.000
15  docker_002    trace.from_os_022.remoteprocess.error_rate             30  2020-05-23T01:30:00+08:00  0.000000  0.000000  0.000   0.000000      0.000
16  docker_002                     trace.self.csf.error_rate             30  2020-05-23T01:30:00+08:00  0.000000  0.000000  0.000   0.000000      0.000
17  docker_002               trace.self.flyremote.error_rate             30  2020-05-23T01:30:00+08:00  0.000000  0.000000  0.000   0.000000      0.000
18  docker_002                    trace.self.jdbc.error_rate             30  2020-05-23T01:30:00+08:00  0.000000  0.000000  0.000   0.000000      0.000
19  docker_002                   trace.self.local.error_rate             30  2020-05-23T01:30:00+08:00  0.000000  0.000000  0.000   0.000000      0.000, 208)```
```